In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

# Load Dataset
(X_train, _), (X_test, _) = mnist.load_data()

# Normalize and Flatten
X_train = X_train.reshape(-1, 784) / 255.0
X_test = X_test.reshape(-1, 784) / 255.0

# PCA
pca = PCA(n_components=32)
pca.fit(X_train)

X_test_pca = pca.transform(X_test)
X_test_pca_recon = pca.inverse_transform(X_test_pca)

pca_mse = mean_squared_error(X_test, X_test_pca_recon)

# Autoencoder
input_img = Input(shape=(784,))
encoded = Dense(32, activation='relu')(input_img)
decoded = Dense(784, activation='sigmoid')(encoded)

autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(
    X_train,
    X_train,
    epochs=5,
    batch_size=256,
    verbose=1
)

# Reconstruction
X_test_ae = autoencoder.predict(X_test)
ae_mse = mean_squared_error(X_test, X_test_ae)

# Results
print('\nPCA Reconstruction Error:', pca_mse)
print('Autoencoder Reconstruction Error:', ae_mse)

if ae_mse < pca_mse:
    print('Autoencoder performs better.')
else:
    print('PCA performs better.')